# J4 après-midi : du script monolithique au DAO

Livret de TP J4 après-midi, **exercice 3**. Le script ci-dessous fonctionne, mais tout y est
mélangé : la connexion, les requêtes, les calculs, l'affichage. On le range en deux classes :
`Restaurant` (une fiche, en objet Python) et `RestaurantDAO` (toutes les requêtes MongoDB).

## Le script de départ (donné, à exécuter tel quel)

In [ ]:
import os
from pymongo import MongoClient

client = MongoClient(os.environ["MONGO_URI"])
collection = client["nyceats"]["restaurants"]
print("Restaurants :", collection.count_documents({}))

# Les restaurants français de Brooklyn et leur score moyen, calculé ici en Python
for doc in collection.find({"cuisine": "French", "borough": "Brooklyn"}).sort("name", 1).limit(5):
    scores = [inspection["score"] for inspection in doc["grades"] if inspection["score"] is not None]
    moyenne = sum(scores) / len(scores) if scores else None
    print(doc["name"], "|", doc["borough"], "|", moyenne)

# Le nombre de restaurants par quartier : 25 359 documents rapatriés pour compter
totaux = {}
for doc in collection.find({}, {"borough": 1}):
    totaux[doc["borough"]] = totaux.get(doc["borough"], 0) + 1
print(totaux)

## 1. La classe `Restaurant`

Une `dataclass` : Python écrit pour nous le constructeur et l'affichage. Complétez la méthode
`depuis_document`, qui fabrique un `Restaurant` à partir d'un document MongoDB, et la propriété
`score_moyen` (`None` s'il n'y a aucun score).

In [ ]:
from dataclasses import dataclass, field


@dataclass
class Restaurant:
    restaurant_id: str
    nom: str
    quartier: str
    cuisine: str
    scores: list = field(default_factory=list)

    @classmethod
    def depuis_document(cls, document):
        # À COMPLÉTER : les scores non vides du tableau grades
        return cls(...)

    @property
    def score_moyen(self):
        # À COMPLÉTER
        ...


katz = Restaurant.depuis_document(collection.find_one({"restaurant_id": "40732665"}))
print(katz)
print(katz.score_moyen)

## 2. La classe `RestaurantDAO`

Toutes les requêtes MongoDB vont ici, et nulle part ailleurs. Chaque méthode répond à une
question métier et renvoie des objets Python (des `Restaurant`, un dictionnaire), jamais un
curseur. Les calculs sur toute la collection se font **dans la base** (pipeline).

In [ ]:
class RestaurantDAO:
    def __init__(self, collection):
        self.collection = collection

    def trouver(self, restaurant_id):
        """Un Restaurant, ou None s'il n'existe pas."""
        # À COMPLÉTER
        ...

    def par_cuisine(self, cuisine, quartier=None, limite=5):
        """Les restaurants d'une cuisine (et d'un quartier), par ordre alphabétique."""
        # À COMPLÉTER
        ...

    def compter_par_quartier(self):
        """{quartier: nombre de restaurants}, calculé par MongoDB."""
        # À COMPLÉTER
        ...

    def pires_cuisines(self, quartier, nombre=3, minimum=100):
        """Les cuisines au score moyen le plus HAUT (le pire) dans un quartier."""
        # À COMPLÉTER
        ...

## 3. Le programme principal

Il ne contient plus aucune requête : il appelle le DAO et affiche. Réécrivez le script de
départ avec le DAO (même affichage), puis affichez les 3 pires cuisines de Manhattan.

In [ ]:
dao = RestaurantDAO(collection)

# À COMPLÉTER : les 5 restaurants français de Brooklyn et leur score moyen

# À COMPLÉTER : le nombre de restaurants par quartier

# À COMPLÉTER : les 3 pires cuisines de Manhattan

## 4. Et si la fiche n'existe pas ?

In [ ]:
# À COMPLÉTER : que renvoie dao.trouver("00000000") ?